[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/19_kubernetes_helm/19_kubernetes_helm.ipynb)

# Módulo 19 · Kubernetes + Helm: la plataforma de observabilidad como código

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 5 h (ruta núcleo ≈ 3 h 30: §1-4, §6, §8-9 y §12) |
| **Servicios de Compose** | Ninguno del stack. Necesita `helm` (4.x) y, para los experimentos con contenedores, **Docker**. Un clúster con kind/k3d es lo ideal en TU máquina (ver «Honestidad» en §3) |
| **RAM aproximada** | ≈ 1-2 GB mientras corre un experimento (API server de k3s + Tempo/Loki/Prometheus/Collector en contenedores efímeros) |
| **Modo sin helm/Docker** | Se ejecuta con los resultados pregrabados (medidos al construir el módulo); la parte conceptual, `deep_merge` y el HPA corren en cualquier sitio |
| **Prerrequisitos** | Módulos 05 (Collector), 17-18 (profiling, muestreo: aquí se despliega el Collector de dos capas del 18) y 16 (SLOs) |

> 🧭 **Ruta núcleo vs ampliación.** Núcleo: topología y Helm (§1-2), render y validación (§4-6), flujo real de telemetría (§8) y afinidad (§9). Ampliación: API server real (§7), autoscaling (§10), Operator (§11), GitOps (§12).

## Objetivos de aprendizaje

1. **Diseñar** la topología de observabilidad en Kubernetes (agente DaemonSet → gateway Deployment con tail sampling → Tempo/Loki/Prometheus → Grafana) y **justificar** cada capa.
2. **Explicar y reproducir** la semántica de *values* de Helm (mapas se fusionan, listas se reemplazan, `null` borra) y escribir un chart propio con esquema, sondas, seguridad y HPA.
3. **Renderizar y validar** con `helm lint`, `helm template`, `kubeconform` y los binarios oficiales (otelcol, Tempo, Loki, promtool) las configuraciones que producen los charts.
4. **Verificar** con un API server real de Kubernetes que los manifiestos y los CRs del OpenTelemetry Operator son válidos, y **distinguir** lo que se probó de lo que no.
5. **Comprobar** de extremo a extremo que la telemetría fluye con las configuraciones renderizadas y **medir** por qué el tail sampling exige afinidad por `traceID` al escalar.
6. **Dimensionar** el autoscaling (HPA) de un componente con estado y **elegir** entre Helm y Operator con criterio.
7. **Aplicar** configuración como código (capas de values, versiones fijadas, CI de validación, despliegues reproducibles).

## 🔁 Conexión con módulos anteriores

<details>
<summary>1. (Módulo 05/18) ¿Por qué el `tail_sampling` obliga a que todos los spans de una traza lleguen a la misma réplica?</summary>

Porque decide sobre la traza **completa** que guarda en memoria; si una réplica solo ve trozos decide con información parcial (p. ej. no ve el span en error o la duración total). En el módulo 18 lo vimos en el simulador; aquí lo **medimos** con dos réplicas reales.

</details>

<details>
<summary>2. (Módulo 00/10) ¿Qué ventaja tiene que los dashboards y datasources sean archivos en el repositorio?</summary>

Son reproducibles, revisables y versionados (diff en un PR). Helm lleva la misma idea al despliegue: `values.yaml` es la fuente de verdad y `helm template` es determinista.

</details>

<details>
<summary>3. (Módulo 16) ¿Qué es el presupuesto de error y por qué importa al actualizar un Collector?</summary>

Cada actualización puede perder telemetría o romper trazas; el despliegue gradual (PDB, `maxUnavailable`, ventana de estabilización del HPA) es cómo se protege el SLO de la propia plataforma.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, copy, gzip, json, math, os, pathlib, subprocess, time
import yaml
from IPython.display import Markdown, display
from supportops import k8s as K
from supportops import sampling as S
K8S = ROOT / "19_kubernetes_helm" / "k8s"
VALUES = K8S / "values"
REC = json.load(gzip.open(ROOT / "19_kubernetes_helm" / "recorded" / "capture_seed42.json.gz", "rt", encoding="utf-8"))   # medido con helm, k3s y los binarios reales (modo degradado)
TOOLS = K.tool_versions()
HELM = TOOLS["helm"] != "—"
DOCKER = S.docker_ok()
CHARTS_OK = HELM and all(K.chart_path(k, pull=True) for k in K.CHARTS)       # los 5 charts oficiales + los 2 propios, en local
LIVE_RENDER = HELM and CHARTS_OK
print(f"helm: {TOOLS['helm']} · docker: {DOCKER} · charts disponibles en local: {CHARTS_OK}")
if not LIVE_RENDER:
    print("→ modo degradado: lo que necesita helm o los charts usa los resultados pregrabados (mismo código, medido al construir el módulo).")

PLAN = [("otel-gateway", "otel-collector", ["otel-gateway.yaml"], "observability"), ("otel-agent", "otel-collector", ["otel-agent.yaml"], "observability"),
        ("tempo", "tempo", ["tempo.yaml"], "observability"), ("loki", "loki", ["loki.yaml"], "observability"), ("grafana", "grafana", ["grafana.yaml"], "observability"),
        ("prometheus", "prom-lab", [], "observability"), ("supportops", "supportops", ["supportops-lab.yaml"], "supportops")]

## 1 · Intuición: dónde vive cada pieza

En `docker compose` todo corre en una máquina. En producción, la plataforma de observabilidad es **un conjunto de cargas de trabajo** con necesidades distintas:

| Pieza | Forma en K8s | Por qué |
|---|---|---|
| **Agente** del Collector | **DaemonSet** (un pod por nodo) | recibe OTLP de los pods del nodo, añade metadatos de K8s (`k8s_attributes`) y reenvía; barato y sin estado |
| **Gateway** del Collector | **Deployment** + HPA + PDB | aquí vive el `tail_sampling` (estado en memoria) y las métricas RED (`span_metrics`); se escala con cuidado |
| **Tempo / Loki** | **StatefulSet** | estado en disco (o almacenamiento de objetos) |
| **Prometheus** | Deployment/StatefulSet | TSDB local; raspa por DNS/descubrimiento |
| **Grafana** | Deployment | sin estado (datasources y dashboards como código) |
| **SupportOps** | Deployment + HPA | la carga que se observa |

> 💡 **Intuición (SRE).** Es el patrón «agente + gateway» de cualquier pipeline de telemetría: el borde es tonto y cercano a la aplicación; el centro es inteligente, centralizado y pequeño. Y como Helm describe todo **declarativamente**, el despliegue de la plataforma es reproducible y revisable igual que el código del agente.

In [ ]:
fig, ax = viz.canvas(12.5, 5.6, xlim=(0, 125), ylim=(0, 56))
ax.text(62, 53, "Topología de observabilidad en Kubernetes (namespaces observability y supportops)", ha="center", fontsize=12, fontweight="bold")
pods = [viz.box(ax, 3, 30 + 7 * i, 17, 5.5, f"SupportOps pod {i+1}", viz.PALETTE["traces"], fontsize=8) for i in range(3)]
agent = viz.box(ax, 28, 31, 19, 14, "otel-agent\nDaemonSet\nk8s_attributes\nload_balancing(traceID)", viz.PALETTE["profiles"], fontsize=8)
gw = viz.box(ax, 56, 31, 21, 14, "otel-gateway ×N\nDeployment+HPA+PDB\nspan_metrics → tail_sampling", viz.PALETTE["metrics"], fontsize=8)
tempo = viz.box(ax, 90, 42, 16, 6, "Tempo (StatefulSet)", viz.PALETTE["traces"], fontsize=8)
loki = viz.box(ax, 90, 33, 16, 6, "Loki (StatefulSet)", viz.PALETTE["logs"], fontsize=8)
prom = viz.box(ax, 90, 24, 16, 6, "Prometheus", viz.PALETTE["metrics"], fontsize=8)
graf = viz.box(ax, 110, 33, 13, 6, "Grafana", viz.PALETTE["neutral"], fontsize=8)
for p in pods: viz.arrow(ax, viz.right(p), (27.5, 38), "", viz.PALETTE["traces"], rad=0.05)
viz.arrow(ax, viz.right(agent), viz.left(gw), "OTLP + hash(traceID)", viz.PALETTE["profiles"])
viz.arrow(ax, viz.right(gw), viz.left(tempo), "trazas muestreadas", viz.PALETTE["traces"], rad=-0.2)
viz.arrow(ax, viz.right(gw), viz.left(loki), "logs", viz.PALETTE["logs"])
viz.arrow(ax, viz.right(gw), viz.left(prom), "métricas RED", viz.PALETTE["metrics"], rad=0.2)
for b in (tempo, loki, prom): viz.arrow(ax, viz.right(b), viz.left(graf), "", viz.PALETTE["neutral"], rad=0.0)
op = viz.box(ax, 28, 8, 49, 9, "OpenTelemetry Operator (opcional): CRDs OpenTelemetryCollector / Instrumentation", viz.PALETTE["neutral"], fontsize=8)
viz.arrow(ax, viz.top(op), (66, 30.5), "gestiona", viz.PALETTE["neutral"], dashed=True)
hpa = viz.box(ax, 90, 8, 33, 9, "Helm: values.yaml por componente\n(configuración como código, versionada)", viz.PALETTE["ok"], fontsize=8)
plt.show()

> 👀 **Qué debes observar**
>
> - Dos capas con **responsabilidades distintas**: el agente es local y sin estado; el gateway concentra el estado (tail sampling) y por eso el agente reparte **por `traceID`** hacia sus réplicas (§9).
> - Las métricas RED se calculan en el gateway **antes** del muestreo (módulo 18); lo que llega a Tempo es solo lo conservado.
> - El Operator es una alternativa a Helm para gestionar el Collector (§11); no sustituye a Tempo/Loki/Grafana.

## 2 · Helm por dentro: chart, values y la regla de fusión

Un **chart** es una carpeta: `Chart.yaml` (metadatos), `values.yaml` (parámetros por defecto), `values.schema.json` (validación, opcional) y `templates/` (YAML con plantillas Go). `helm template <release> <chart> -f values.yaml` produce los manifiestos;
`helm install` los envía al API server y guarda la **release** (revisión, valores usados) en un Secret del clúster. Veamos el chart propio del curso:

In [ ]:
chart = K8S / "charts" / "supportops"
for p in sorted(chart.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(chart)):38} {len(p.read_text().splitlines()):3} líneas")
print()
print((chart / "values.yaml").read_text()[:900])

### 2.1 · La regla de fusión de *values*, a mano

Cuando pasas `-f base.yaml -f prod.yaml --set x=y`, Helm **fusiona** las capas: los **mapas** se combinan recursivamente, las **listas se reemplazan enteras** y `null` **borra** la clave heredada. Es la fuente de errores nº 1 al personalizar `config.service.pipelines`
o `policies`. La implementamos en 12 líneas (`K.deep_merge`) y comprobamos contra `helm template` real.

In [ ]:
import inspect
print(inspect.getsource(K.deep_merge))
base = {"config": {"processors": {"batch": {}, "memory_limiter": {"check_interval": "5s"}}, "service": {"pipelines": {"logs": {"receivers": ["otlp"], "exporters": ["debug"]}}}}}
over = {"config": {"processors": {"memory_limiter": {"check_interval": "1s"}, "tail_sampling": {"decision_wait": "10s"}}, "service": {"pipelines": {"logs": {"exporters": ["otlp_http/loki"]}, "traces": None}}}}
merged = K.deep_merge(base, over)
print(json.dumps(merged, indent=1))

In [ ]:
if LIVE_RENDER:
    over = {"config": {"service": {"pipelines": {"logs": {"exporters": ["debug", "otlp_http/loki"]}}}}}
    ms = K.render("demo", "otel-collector", [over, {"mode": "deployment", "image": {"repository": "otel/opentelemetry-collector-contrib"}}])
    helm_cfg = yaml.safe_load(K.configmap_text(ms, "demo", "relay"))
    defaults = yaml.safe_load(open(K.chart_path("otel-collector") + "/values.yaml"))
    mine = K.deep_merge(defaults, over)
    print("Chart: pipeline de logs según helm →", helm_cfg["service"]["pipelines"]["logs"])
    print("Mi deep_merge                      →", mine["config"]["service"]["pipelines"]["logs"])
    print("¿Iguales?", helm_cfg["service"]["pipelines"]["logs"] == mine["config"]["service"]["pipelines"]["logs"],
          "· la lista exporters NO se concatenó con [debug] por defecto: se reemplazó")
else:
    print("(sin helm/charts) pregrabado: helm y deep_merge coinciden; las listas se reemplazan")

> 👀 **Qué debes observar**
>
> - `memory_limiter` se **fusionó** (conserva `check_interval` nuevo y los demás campos del chart), `traces: null` **borró** el pipeline de trazas heredado y la lista `exporters` de `logs` fue **reemplazada** (no `["debug", "otlp_http/loki"]` concatenada con la anterior).
> - `helm template` real da exactamente el mismo resultado que nuestra función: ahora sabes qué hace Helm con tus capas de values. Consecuencia práctica: al sobrescribir una lista, **repite todos sus elementos**; al quitar un componente por defecto del chart usa `null`.

## 3 · El entorno, con honestidad

Qué hay en esta máquina y qué se verificó al construir el módulo. Los charts oficiales **no se copian** al curso: se fijan por versión y se descargan de su repositorio (o de un clon de GitHub, como aquí).

In [ ]:
display(pd.Series(TOOLS, name="versión").to_frame())
rows = K.chart_versions() if CHARTS_OK else REC["meta"]["charts"]
display(pd.DataFrame(rows))

> ⚠️ **Qué corrió en un clúster real y qué NO.** En el sandbox donde se construyó el módulo `docker` ejecuta contenedores, pero **ni `kind` (v1.31 y v1.34) ni `k3s` pudieron arrancar pods**: `runc` anidado falla con
> `can't get final child's PID from pipe: EOF`. Por tanto **ningún pod llegó a ejecutarse en un clúster**. Lo que SÍ se verificó con binarios reales:
>
> | Verificado | Cómo |
> |---|---|
> | Manifiestos y values de los 7 *releases* | `helm lint`, `helm template`, `kubeconform` (esquemas de K8s 1.31) |
> | Las **configuraciones que renderizan los charts** | `otelcol-contrib validate`, `tempo -config.verify`, `loki -verify-config`, `promtool check config` |
> | Objetos y CRs contra un **API server real** (k3s 1.31) | `helm install`, `kubectl apply --dry-run=server`, rechazo de CRs inválidos por el esquema del CRD |
> | **Flujo de telemetría de extremo a extremo** | Tempo, Loki, Prometheus y el gateway en contenedores con **esas mismas configuraciones** y los DNS de los Services como alias de red |
> | **Afinidad por `traceID`** con 2 réplicas del gateway | Collectors reales (`load_balancing`) y comparación de trazas completas |
>
> **No** verificado: planificación y ciclo de vida de pods, sondas (también la `livenessProbe` por latido del chart `supportops`), el Service `otel-agent` con `internalTrafficPolicy: Local`, `hostPort`, HPA con métricas reales (no hay metrics-server), el Operator en ejecución (sus imágenes están en `ghcr.io`/`quay.io`, inaccesibles aquí), el webhook de admisión y la inyección de auto-instrumentación.
> En tu máquina con kind o k3d todo eso se ejecuta con los mismos archivos (`k8s/kind-config.yaml`, `k8s/values/`).

## 4 · Renderizar y validar: `helm template` + `helm lint` + `kubeconform`

Para cada *release* (6 en `observability`, 1 en `supportops`) renderizamos los manifiestos con los values del módulo, pasamos `helm lint` y validamos cada documento contra el esquema OpenAPI de Kubernetes con `kubeconform`.

In [ ]:
if LIVE_RENDER:
    REN, rows = {}, []
    for rel, ch, vals, ns in PLAN:
        files = [VALUES / v for v in vals]
        ms = K.render(rel, ch, files, namespace=ns)
        REN[rel] = ms
        ok, _ = K.lint(ch, files)
        kc = K.kubeconform(ms)
        req = K.resource_requests(ms)
        rows.append({"release": rel, "chart": ch, "manifests": len(ms), "kinds": dict(K.kinds_count(ms)), "lint": ok, "kubeconform_valid": kc["valid"], "kubeconform_invalid": kc["invalid"], "cpu_m": req["cpu_m"], "memory_mib": req["memory_mib"]})
else:
    REN, rows = {}, REC["render"]
R = pd.DataFrame(rows)
display(R[["release", "chart", "manifests", "lint", "kubeconform_valid", "kubeconform_invalid", "cpu_m", "memory_mib"]])
print("total de manifiestos:", int(R["manifests"].sum()), "· inválidos para el esquema de K8s:", int(R["kubeconform_invalid"].sum()))

In [ ]:
kinds = collections.Counter()
per_rel = {r["release"]: r["kinds"] for r in rows}
all_kinds = sorted({k for d in per_rel.values() for k in d}, key=lambda k: -sum(d.get(k, 0) for d in per_rel.values()))
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
bottom = np.zeros(len(per_rel))
cmap = plt.cm.tab20(np.linspace(0, 1, len(all_kinds)))
for k, c in zip(all_kinds, cmap):
    vals = np.array([per_rel[r].get(k, 0) for r in per_rel])
    axs[0].bar(list(per_rel), vals, bottom=bottom, color=c, label=k); bottom += vals
axs[0].set_ylim(0, bottom.max() + 4); axs[0].set_title("Objetos de Kubernetes por release"); axs[0].set_ylabel("nº de manifiestos"); axs[0].legend(fontsize=7, ncol=3, loc="upper right"); axs[0].tick_params(axis="x", labelrotation=30, labelsize=8)
NODE_CPU, NODE_MEM = 4000, 8 * 1024
cpu = R["cpu_m"].values; mem = R["memory_mib"].values
xr = np.arange(len(R))
axs[1].bar(xr - 0.2, cpu / NODE_CPU * 100, 0.4, color=viz.PALETTE["metrics"], label="CPU requests (% de 4 vCPU)")
axs[1].bar(xr + 0.2, mem / NODE_MEM * 100, 0.4, color=viz.PALETTE["traces"], label="memoria requests (% de 8 GiB)")
axs[1].set_xticks(xr); axs[1].set_xticklabels(R["release"])
axs[1].set_title(f"Requests por release — total {cpu.sum():.0f} m · {mem.sum():.0f} MiB"); axs[1].set_ylabel("% de un nodo de laboratorio"); axs[1].legend(fontsize=8); axs[1].tick_params(axis="x", labelrotation=30, labelsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **Todos** los manifiestos de los 7 releases pasan `helm lint` y el esquema de Kubernetes (`kubeconform`: 0 inválidos). Loki y el gateway concentran más objetos (cuentas de servicio, RBAC, Services).
> - Las **requests** suman ≈ 0,6 vCPU y ≈ 1,4 GiB: caben holgadas en un nodo kind de laboratorio; en producción se dimensionan con medidas (el módulo 18 te dio el cálculo de memoria del tail sampling).
> - Los **límites** de memoria están fijados y los de CPU **no** (a propósito: un límite de CPU provoca *throttling* en un Collector bajo ráfagas; ver 🧠).

## 5 · Configuración como código: los values del gateway

El archivo `k8s/values/otel-gateway.yaml` está **generado desde Python** (el módulo lo construye con `build_19.py`) y es la pieza central: reutiliza la política del módulo 18 (errors, slow, costly, baseline), calcula RED antes del muestreo y exporta a los tres backends por los **nombres DNS de los Services**.
Se muestra tal cual, con los puntos clave:

In [ ]:
txt = (VALUES / "otel-gateway.yaml").read_text()
print(txt[:3400])

Puntos clave del archivo (compáralo con el módulo 18):

- `config.receivers.jaeger/zipkin/prometheus: null` → **borra** receptores que el chart trae por defecto (regla de fusión de §2).
- `connectors.span_metrics` + `forward`: las métricas RED salen de `traces/in` **antes** de `tail_sampling` (`traces/sampled`).
- `extraManifests`: el Service **headless** `otel-gateway-headless` que el agente usa para repartir por `traceID`.
- `autoscaling.behavior.scaleDown`: ventana de estabilización de 10 min y bajada de 1 pod cada 2 min: escalar **hacia abajo** un componente con estado es lo delicado (§10).
- `resources.limits`: solo memoria; `podDisruptionBudget.minAvailable: 1`.

## 6 · Validar lo que renderizan los charts con los binarios oficiales

Un chart puede ser YAML perfecto y producir una **configuración de aplicación** inválida (el Collector, Tempo, Loki y Prometheus tienen su propio esquema). Extraemos los ConfigMaps renderizados y los validamos con el binario de cada uno (en contenedores efímeros, sin red).

In [ ]:
if LIVE_RENDER and DOCKER:
    cfgs = {"otel-gateway": ("collector", K.configmap_text(REN["otel-gateway"], "otel-gateway", "relay")), "otel-agent": ("collector", K.configmap_text(REN["otel-agent"], "otel-agent", "relay")),
            "tempo": ("tempo", K.configmap_text(REN["tempo"], "tempo", "tempo.yaml")), "loki": ("loki", K.configmap_text(REN["loki"], "loki", "config.yaml")),
            "prometheus": ("prometheus", K.configmap_text(REN["prometheus"], "prometheus", "prometheus.yml"))}
    BIN = {k: K.validate_with_binary(kind, txt)[0] for k, (kind, txt) in cfgs.items()}
else:
    BIN = REC["binary_validation"]
display(pd.Series({k: ("✅ válida" if v else "❌") for k, v in BIN.items()}, name="validación con el binario oficial").to_frame())

Dos errores **reales** que aparecieron al construir los values, tal como los reportó el binario:

1. `load_balancing.resolver.dns.port` debe ser una **cadena** (`"4317"`), no un entero. Con `port: 4317` el Collector 0.162.0 responde: `'port' expected type 'string', got unconvertible type 'int'`.
2. El preset `kubernetesAttributes` usa `node_from_env_var: K8S_NODE_NAME`: la configuración **depende de variables de entorno que inyecta el pod** (el chart las pone con *downward API*). Validar fuera del pod exige definirlas.

In [ ]:
if LIVE_RENDER and DOCKER:
    bad = K.configmap_text(REN["otel-agent"], "otel-agent", "relay").replace('port: "4317"', "port: 4317")
    print(K.validate_with_binary("collector", bad)[1][-330:])
else:
    print(REC["agent_port_int"])

> 👀 **Qué debes observar**
>
> - Las 5 configuraciones renderizadas son válidas **según el binario**, no solo según el esquema de YAML. Es el control que falta en la mayoría de pipelines de CI de Helm: `helm lint` no sabe qué es un `tail_sampling`.
> - Los dos fallos de arriba se detectaron **antes** de desplegar. En un clúster habrían sido un `CrashLoopBackOff` con el error en los logs del pod.

## 7 · Un API server real: `helm install` y CRs del Operator

Con `k3s` (Kubernetes real) en un contenedor tenemos API server, controladores, admisión, CRDs y RBAC. Instalamos los 7 releases con `helm upgrade --install` (sin esperar a los pods: no pueden arrancar aquí), verificamos primero con `--dry-run=server`
(el API server **valida** de verdad) y aplicamos los CRs del Operator contra los CRDs reales instalados por su chart.

In [ ]:
if LIVE_RENDER and DOCKER:
    with K.ApiServerLab() as lab:
        for ns in ("observability", "supportops"):
            lab.ensure_namespace(ns)
        ops = lab.helm_install("otel-operator", "otel-operator", [K8S / "operator" / "values-operator.yaml"], namespace="opentelemetry-operator-system", set_values={"admissionWebhooks.create": "false"})
        API = {"operator_install_rc": ops.returncode, "crds": sorted(x.split("/")[-1] for x in lab.kubectl("get", "crd", "-o", "name").stdout.split() if "opentelemetry" in x), "installs": {}}
        for rel, ch, vals, ns in PLAN:
            dry = lab.helm_install(rel, ch, [VALUES / v for v in vals], namespace=ns, dry_run_server=True)
            real = lab.helm_install(rel, ch, [VALUES / v for v in vals], namespace=ns)
            API["installs"][rel] = {"dry_run_server_rc": dry.returncode, "install_rc": real.returncode}
        API["objects"] = dict(lab.objects("observability"))
        API["cr_collector"] = lab.apply((K8S / "operator" / "collector-gateway.yaml").read_text(), dry_run=True).stdout.strip()
        API["cr_instrumentation"] = lab.apply((K8S / "operator" / "instrumentation.yaml").read_text(), dry_run=True).stdout.strip()
        API["cr_invalid"] = lab.apply((K8S / "operator" / "collector-gateway.yaml").read_text().replace("replicas: 2", 'replicas: "dos"'), dry_run=True).stderr.strip()[:300]
        API["hpa"] = lab.kubectl("get", "hpa", "-n", "observability", "-o", "wide").stdout
        ok, why = lab.can_run_pods(timeout_s=45)
        API["can_run_pods"] = {"ok": ok, "reason": why}
else:
    API = REC["api"]
display(pd.DataFrame(API["installs"]).T.rename(columns={"dry_run_server_rc": "dry-run=server (rc)", "install_rc": "install (rc)"}))
print("CRDs del Operator instalados por su chart:", API["crds"])
print("CR OpenTelemetryCollector →", API["cr_collector"]); print("CR Instrumentation        →", API["cr_instrumentation"])
print("CR inválido (replicas: \"dos\") →", API["cr_invalid"].splitlines()[0] if API["cr_invalid"] else "")
print("HPA creado (sin metrics-server → TARGETS <unknown>):\n" + API["hpa"])
print("¿Pueden arrancar pods en este sandbox?", API["can_run_pods"])

In [ ]:
objs = pd.Series(API["objects"]).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(objs.index, objs.values, color=viz.PALETTE["traces"]); ax.set_title("Objetos creados en el namespace observability (API server real)"); ax.set_xlabel("nº de objetos")
for i, v in enumerate(objs.values): ax.text(v, i, f" {v}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
cols = ["helm lint", "kubeconform", "config con\nsu binario", "dry-run=server", "helm install", "pods\nRunning"]
rel_names = [r["release"] for r in rows]
M = np.full((len(rel_names), len(cols)), np.nan)
for i, r in enumerate(rows):
    M[i, 0] = float(r["lint"]); M[i, 1] = float(r["kubeconform_invalid"] == 0)
    if r["release"] in BIN: M[i, 2] = float(BIN[r["release"]])
    ins = API["installs"].get(r["release"], {})
    M[i, 3] = float(ins.get("dry_run_server_rc", 1) == 0); M[i, 4] = float(ins.get("install_rc", 1) == 0)
    M[i, 5] = float(API["can_run_pods"]["ok"])
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(9, 3.9))
ax.imshow(np.ma.masked_invalid(M), cmap=ListedColormap([viz.PALETTE["error"], viz.PALETTE["ok"]]), vmin=0, vmax=1, aspect="auto")
ax.set_facecolor("#d9dde3"); ax.grid(False)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=8); ax.set_yticks(range(len(rel_names))); ax.set_yticklabels(rel_names, fontsize=8)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, "n/a" if np.isnan(M[i, j]) else ("✔" if M[i, j] else "✘"), ha="center", va="center", color=("#1f2937" if np.isnan(M[i, j]) else "white"), fontsize=11, fontweight="bold")
ax.set_title("Matriz de verificación por release (gris = no aplica; rojo = NO se pudo en este sandbox)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **Todo se instala** (`rc = 0`, también con `--dry-run=server`): el API server aceptó los 7 releases y los 2 CRs del Operator, e instaló los CRDs que su chart trae **plantillados** (`crds.create`).
> - Un CR con `replicas: "dos"` es **rechazado por el esquema del CRD** con un mensaje claro: validación gratuita antes del despliegue.
> - El HPA se crea pero marca `<unknown>/70%`: sin `metrics-server` y sin pods ejecutándose no hay métrica. Y la última línea es la verdad incómoda del sandbox: **los pods no arrancan** (runc anidado), por eso §8-9 usan contenedores directos.

## 8 · El flujo de telemetría de extremo a extremo con las configuraciones renderizadas

Como no hay pods, levantamos Tempo, Loki, Prometheus y el gateway en **contenedores con las configuraciones que acaban de renderizar los charts** (`configmap_files`) y con los DNS de los Services de Kubernetes como alias de red
(`tempo.observability.svc.cluster.local`…): lo que el chart pone en `endpoint:` resuelve igual que en el clúster. Enviamos tráfico real de SupportOps (trazas, logs y métricas por OTLP) y consultamos cada backend.

In [ ]:
if LIVE_RENDER and DOCKER:
    from supportops import DEFAULT_FAULT_RATES, SupportAgent, generate_tickets, run_tickets
    from supportops.telemetry import OTelInstrumentor, setup_telemetry
    lab_cm = dict(tempo=K.configmap_files(REN["tempo"], "tempo"), loki=K.configmap_files(REN["loki"], "loki"), loki_runtime=K.configmap_files(REN["loki"], "loki-runtime"),
                  prometheus=K.configmap_files(REN["prometheus"], "prometheus"), gateway=K.configmap_files(REN["otel-gateway"], "otel-gateway"))
    with K.FlowLab(lab_cm["tempo"], lab_cm["loki"], lab_cm["loki_runtime"], lab_cm["prometheus"], lab_cm["gateway"], gateway_replicas=1) as lab:
        tel = setup_telemetry("supportops-agent", endpoint=lab.gateway_endpoints[0], mode="otlp", resource_attrs={"deployment.environment.name": "lab", "k8s.pod.name": "sim"}, metric_interval_s=3)
        agent = SupportAgent(instrumentor=OTelInstrumentor(tel))
        tk = generate_tickets(300, seed=SEED, duration_s=60, start_s="live", fault_rates=DEFAULT_FAULT_RATES)
        res = run_tickets(agent, tk); tel.force_flush(10); time.sleep(4); tel.force_flush(10)
        lab.wait_until(lambda: len(lab.tempo_trace_ids()) >= 20, 90); time.sleep(5)
        FLOW = {"tickets": len(tk), "traces_in_tempo": len(lab.tempo_trace_ids()), "loki_lines": lab.wait_until(lambda: lab.loki_lines() > 100 and lab.loki_lines(), 60) or 0,
                "prom_genai_series": len(lab.wait_until(lambda: lab.prom_query("gen_ai_client_inference_duration_seconds_count"), 60) or []),
                "span_metrics_series": 0, "span_metrics_calls": 0.0, "gateway_up": [r["value"][1] for r in (lab.wait_until(lambda: lab.prom_query('up{job="otel-gateway"}'), 45) or [])]}
        sm = lab.wait_until(lambda: [r for r in lab.prom_query("traces_span_metrics_calls_total") if float(r["value"][1]) > 0], 90) or []
        FLOW["span_metrics_series"], FLOW["span_metrics_calls"] = len(sm), sum(float(r["value"][1]) for r in sm)
        FLOW["dashboard_queries_series"] = {}
        for k, e in {"replicas": 'count(up{job="otel-gateway"} == 1)', "accepted": 'sum(increase(otelcol_receiver_accepted_spans{job="otel-gateway"}[10m]))',
                     "on_memory": 'sum by (instance) (otelcol_processor_tail_sampling_sampling_traces_on_memory{job="otel-gateway"})'}.items():
            FLOW["dashboard_queries_series"][k] = len(lab.prom_query(e))
        tel.shutdown()
else:
    FLOW = REC["flow"]
display(pd.Series({k: v for k, v in FLOW.items() if k != "dashboard_queries_series"}, name="resultado").to_frame())
print("consultas del dashboard 19 contra el Prometheus del lab (nº de series):", FLOW["dashboard_queries_series"])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 3.8))
funnel = {"tickets atendidos": FLOW["tickets"], "trazas en Tempo\n(tail sampling)": FLOW["traces_in_tempo"], "líneas de log en Loki": FLOW["loki_lines"]}
axs[0].bar(list(funnel), list(funnel.values()), color=[viz.PALETTE["neutral"], viz.PALETTE["traces"], viz.PALETTE["logs"]])
for i, v in enumerate(funnel.values()): axs[0].text(i, v, f"{v}", ha="center", va="bottom")
axs[0].set_title("Lo que sale de SupportOps y lo que llega a cada backend"); axs[0].set_ylabel("nº")
sm = {"series gen_ai.* (Prometheus)": FLOW["prom_genai_series"], "series span_metrics": FLOW["span_metrics_series"], "réplicas del gateway con up=1": len(FLOW["gateway_up"])}
axs[1].barh(list(sm), list(sm.values()), color=viz.PALETTE["metrics"]); axs[1].set_title("Métricas: del SDK y generadas por el gateway")
plt.tight_layout(); plt.show()
print(f"Fracción de trazas conservadas: {FLOW['traces_in_tempo'] / FLOW['tickets']:.0%} (la política del módulo 18 conserva lo interesante + un 10 % base)")

> 👀 **Qué debes observar**
>
> - Con la configuración **tal como la renderizan los charts**, la telemetría fluye de extremo a extremo: ≈ 1/5 de las trazas llega a Tempo (tail sampling), los **logs** completos a Loki (con `trace_id` como structured metadata) y las **métricas** del SDK y las RED del gateway a Prometheus.
> - `up{job="otel-gateway"} = 1`: el Prometheus de `prom-lab` descubre las réplicas **por DNS** del Service headless, igual que lo haría en el clúster.
> - Las consultas del dashboard `19-kubernetes.json` devuelven series con datos reales (se validan en esta misma celda).

## 9 · Afinidad por `traceID`: por qué el gateway no se puede escalar «a ciegas»

Con **2 réplicas** del gateway detrás de un Service normal (reparto por turnos), cada réplica recibe *trozos* de cada traza (los spans de servicios distintos llegan en lotes distintos) y decide con información parcial. La solución (§1): el agente usa el exportador **`load_balancing`** con `routing_key: traceID`
y un resolvedor DNS sobre el Service **headless**. Lo medimos con dos Collectors reales (`tail_sampling` → archivo) y las mismas 600 trazas en ambos modos.

In [ ]:
if DOCKER:
    corpus = S.build_corpus(600)
    AFF = K.affinity_experiment(corpus.traces, replicas=2)
else:
    AFF = REC["affinity"]
df = pd.DataFrame(AFF).T[["traces_kept", "complete_pct", "interesting_recall_pct", "interesting_complete_pct", "spans_kept"]].round(1)
display(df)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
x = np.arange(2); w = 0.35
ax.bar(x - w / 2, df["complete_pct"], w, color=viz.PALETTE["warn"], label="% de trazas conservadas COMPLETAS")
ax.bar(x + w / 2, df["interesting_complete_pct"], w, color=viz.PALETTE["error"], label="% de las trazas interesantes conservadas COMPLETAS")
ax.set_xticks(x); ax.set_xticklabels(df.index, fontsize=9); ax.set_ylim(0, 110); ax.set_ylabel("%"); ax.set_title("Sin afinidad, el tail sampling guarda trazas rotas"); ax.legend(fontsize=8)
for i, (a, b) in enumerate(zip(df["complete_pct"], df["interesting_complete_pct"])): ax.text(i - w / 2, a + 1, f"{a:.0f}", ha="center"); ax.text(i + w / 2, b + 1, f"{b:.0f}", ha="center")
plt.show()

> 👀 **Qué debes observar**
>
> - **Sin afinidad**, solo ≈ 2/3 de las trazas conservadas están completas y, lo peor, **solo ≈ 42 % de las interesantes** (errores, lentas): una réplica guarda el span en error y la otra no, o decide «no lenta» al ver media traza. **Con `load_balancing` por `traceID`**: el **100 %**.
> - El número de trazas conservadas es el mismo (127): lo que cambia es su **calidad**. Un panel de «trazas con error» cuenta bien, pero al abrir la traza faltan spans: el síntoma más desconcertante del tail sampling mal desplegado.
> - En el clúster el resolvedor `dns` sobre `otel-gateway-headless` (una A por pod) mantiene el anillo al día cada `interval`; el chart del agente lo configura así (`k8s/values/otel-agent.yaml`). El resolvedor `k8s` (endpoints) requiere RBAC adicional.

## 10 · Autoscaling del gateway

El HPA calcula $\text{réplicas}_{\text{deseadas}}=\left\lceil \text{réplicas}_{\text{actuales}}\cdot\dfrac{\text{métrica actual}}{\text{objetivo}}\right\rceil$ (con una tolerancia del 10 %: si la razón está entre 0,9 y 1,1 no hace nada), acotado a `[min, max]`.
Subir réplicas es sencillo; **bajarlas** en un componente con estado es delicado: cada cambio de réplicas **reasigna** parte de las trazas en curso (el anillo del `load_balancing` cambia) y las trazas partidas se rompen. De ahí `behavior.scaleDown` con una ventana de estabilización larga.

In [ ]:
rng = np.random.default_rng(SEED)
t = np.arange(0, 120)                      # 120 pasos de 15 s = 30 min
load = 150 + 30 * np.sin(t / 10) + rng.normal(0, 8, t.size)
load[40:70] += 420                          # un incidente: tráfico ×4 durante 7,5 min
CAP = 100                                   # unidades de trabajo por réplica a 100 % de CPU
fast = K.hpa_simulate(load, CAP, 0.7, min_replicas=2, max_replicas=10, step_s=15, scale_down_window_s=15)
slow = K.hpa_simulate(load, CAP, 0.7, min_replicas=2, max_replicas=10, step_s=15, scale_down_window_s=600)
fig, axs = plt.subplots(2, 1, figsize=(10, 5.4), sharex=True)
axs[0].plot(t * 15 / 60, load, color=viz.PALETTE["neutral"], lw=1.5); axs[0].set_ylabel("carga (trabajo/s)"); axs[0].set_title("Un pico de tráfico y la respuesta del HPA")
axs[1].step(t * 15 / 60, fast["replicas"], where="post", color=viz.PALETTE["error"], lw=2, label="ventana de bajada 15 s (sin estabilización)")
axs[1].step(t * 15 / 60, slow["replicas"], where="post", color=viz.PALETTE["ok"], lw=2, label="ventana de bajada 10 min (como los values)")
axs[1].set_xlabel("minutos"); axs[1].set_ylabel("réplicas"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
changes = lambda r: int(np.sum(np.diff(r["replicas"]) != 0))
print(f"cambios de réplicas: sin estabilización {changes(fast)} · con ventana de 10 min {changes(slow)} · pico {max(slow['replicas'])} réplicas · réplica-minutos {sum(slow['replicas']) * 15 / 60:.0f} vs {sum(fast['replicas']) * 15 / 60:.0f}")

> 👀 **Qué debes observar**
>
> - Ambas curvas **suben igual de rápido** al llegar el pico (el HPA reacciona en un paso); la diferencia está en la **bajada**: sin estabilización oscila siguiendo el ruido de la carga (cada cambio = trazas reasignadas); con la ventana de 10 min mantiene las réplicas y baja de forma ordenada. Cuesta algunas réplica-minutos extra, y las paga el **recall** de trazas íntegras.
> - La métrica de CPU es una aproximación: la presión real del gateway es **memoria** (`sampling_traces_on_memory`). En producción se escala por métricas personalizadas (KEDA / Prometheus Adapter) sobre esa serie; es un reto del proyecto.
> - En este sandbox el HPA **no se ejecutó** (sin metrics-server ni pods): la simulación es el algoritmo documentado, no una medida del clúster.

## 11 · OpenTelemetry Operator: CRDs en lugar de values

El **Operator** (chart 0.126.0, operador 0.161.0) añade CRDs y un controlador: `OpenTelemetryCollector` (v1beta1) crea el Deployment/DaemonSet/StatefulSet, Service, ConfigMap, **HPA y PDB** a partir de un solo CR; `Instrumentation` (v1alpha1) permite
**auto-instrumentar** pods con una anotación (`instrumentation.opentelemetry.io/inject-python: "true"`: un webhook inyecta un *init container* con el SDK). Requiere un certificado para su webhook: cert-manager o, como aquí, `autoGenerateCert`.

In [ ]:
print((K8S / "operator" / "collector-gateway.yaml").read_text()[:1600])
print("...")
print((K8S / "operator" / "values-operator.yaml").read_text())

| | **Helm** (chart del Collector) | **Operator** (CRs) |
|---|---|---|
| Qué gestionas | values → manifiestos (tú ves y versionas el YAML resultante) | un CR → el controlador crea y **reconcilia** los recursos |
| Auto-instrumentación | no | **sí** (`Instrumentation` + anotación) |
| Reconciliación continua | no (Helm aplica y se va) | sí (corrige derivas) |
| Dependencias | ninguna | webhook + certificado (cert-manager) |
| Operar/depurar | `helm template` explica todo | un nivel más de indirección |
| Cuándo | plataformas con GitOps y equipos que quieren ver el YAML | muchos Collectors, TargetAllocator para Prometheus, inyección masiva |

Existe además el chart **`opentelemetry-kube-stack`** (0.24.6, app 0.159.0 en el repositorio verificado), que instala el Operator y colectores preconfigurados «de una tacada». El CR de este módulo **se validó contra el API server real** (§7); su ejecución (imágenes en `ghcr.io`) **no** se pudo probar aquí.

## 12 · Configuración como código y GitOps

1. **Capas de values**: `-f base.yaml -f env/prod.yaml`: la última gana (regla de §2). Mantén `base` mínimo y diferencias por entorno pequeñas.
2. **Versiones fijadas**: chart (`--version`), imagen (`tag` y, mejor, `digest`) y apps. Aquí todas las versiones coinciden con el stack de Compose (Collector 0.162.0, Tempo 3.1.0, Loki 3.7.8, Grafana 13.2.4, Prometheus v3.15.0).
3. **Reinicios por cambio de configuración**: el chart del Collector pone un *checksum* de la ConfigMap en las anotaciones del pod (`enableConfigChecksumAnnotation`): cambiar la configuración cambia el hash y fuerza el *rollout*. Lo comprobamos:

In [ ]:
if LIVE_RENDER:
    def ck(over):
        ms = K.render("otel-gateway", "otel-collector", [VALUES / "otel-gateway.yaml", over])
        dep = next(m for m in ms if m["kind"] == "Deployment")
        return dep["spec"]["template"]["metadata"]["annotations"]
    a0 = ck({})
    a1 = ck({"config": {"processors": {"tail_sampling": {"decision_wait": "20s"}}}})
    a2 = ck({"replicaCount": 3})
    print("checksum base      :", a0)
    print("decision_wait 20 s :", a1, "→ cambia:", a0 != a1)
    print("replicaCount 3     :", a2, "→ no cambia:", a0 == a2)
else:
    print("(sin helm) pregrabado: cambiar la configuración cambia el checksum del pod; cambiar solo las réplicas no")

4. **Pipeline de CI** (cada PR): `helm lint` → `helm template | kubeconform` → validación de las configuraciones renderizadas con los binarios (§6) → `helm install --dry-run=server` contra un clúster efímero (§7) → pruebas de flujo. Todo está en este módulo salvo el último paso en un clúster con pods.
5. **Secretos**: nada de contraseñas en values (aquí `adminPassword: admin` es **solo laboratorio**): `existingSecret`, External Secrets o Sealed Secrets.
6. **GitOps** (Argo CD/Flux): el repositorio es el estado deseado; el controlador aplica y corrige derivas. Los charts de este módulo son compatibles tal cual (`helm template` determinista).

## 13 · 🏭 En producción

- **Grafana Labs** publica charts para Tempo (`tempo-distributed` 3.13.0 verificado) y Loki; en 2026 los charts de la comunidad (Tempo monolítico, Loki, Grafana) se mantienen en `grafana-community/helm-charts` y el repositorio `grafana/helm-charts` quedó para los productos Enterprise: **verifica el repositorio de cada chart** antes de fijarlo.
- El **Collector** se despliega casi siempre en **dos niveles** (agente DaemonSet + gateway Deployment) y el gateway con tail sampling detrás de `load_balancing` (documentado en el README del procesador y usado por Honeycomb Refinery y los despliegues de OpenTelemetry Demo).
- Los charts de **kube-prometheus-stack**, **opentelemetry-kube-stack** y **k8s-monitoring** empaquetan «un stack completo»: útiles para empezar; aprende primero a leer los values de cada pieza (este módulo) para poder depurarlos.
- Las plataformas de ingeniería maduras no despliegan nunca desde una laptop: **GitOps + revisión + validación en CI** (kubeconform, conftest/Kyverno para políticas de seguridad: `runAsNonRoot`, `readOnlyRootFilesystem`, límites de memoria).

## 14 · 🧠 Secretos de la élite

1. **`helm lint` no valida la aplicación.** Pasa con un Collector de configuración inválida. Extrae el ConfigMap renderizado y valídalo con el binario (medido: `port` de `load_balancing` entero ⇒ error de tipo; §6).
2. **Las listas se reemplazan, los mapas se fusionan.** Al sobrescribir `service.pipelines.logs.exporters` repite toda la lista; para quitar un receptor por defecto usa `null` (§2). Es la causa de «he añadido un exportador y han desaparecido los otros».
3. **Un componente con estado no se escala como uno sin estado.** Sin `load_balancing` por `traceID` el 42 % de las trazas interesantes quedaron rotas con solo 2 réplicas (§9); y cada cambio de réplicas reasigna trazas en curso: estabiliza la bajada (§10) y usa PDB.
4. **No pongas límites de CPU a un Collector** (sí de memoria y `memory_limiter` por debajo): el *throttling* de CFS produce latencia y pérdidas justo en las ráfagas. Pide CPU (`requests`) para que el scheduler lo reserve. *Fuente: documentación de Kubernetes («cpu limits are enforced by CPU throttling», `manage-resources-containers.md`, leída en el repositorio); la consecuencia sobre las ráfagas del Collector es razonamiento del curso, no una cita.*
5. **El agente y el gateway tienen presupuestos distintos.** El agente va por nodo: barato (50 m/128 Mi aquí); el gateway concentra memoria (≈ tráfico × `decision_wait`, módulo 18). Dimensiónalos por separado.
6. **La configuración depende del entorno del pod.** `${env:MY_POD_IP}`, `K8S_NODE_NAME`… provienen del *downward API*: una validación fuera del pod necesita las mismas variables (§6).
7. **`--wait` de Helm 4.** `--wait=false` está obsoleto (se sustituye por `--wait=hookOnly`): scripts de CI escritos para Helm 3 emiten avisos o esperan por defecto. Fija la versión de Helm en CI.
8. **Nada de `latest`, nada sin versión.** Un chart sin `--version` y una imagen sin `tag` convierten tu plataforma en una lotería; las 7 versiones de este módulo están verificadas y fijadas.
9. **No todo el pipeline de trazas se reparte por `traceID`: las métricas derivadas tienen otra afinidad.** El tail sampling necesita **todos los spans de una traza** en el mismo Collector (balanceo por `traceID`); pero `span_metrics` y el service graph agrupan por **servicio** y por **pares cliente/servidor**, y la guía de Grafana para escalar el muestreo los coloca en la segunda capa con su propia clave de enrutado. Si calculas RED después de repartir por traza sin pensarlo, cada réplica ve fragmentos de la misma serie o la mitad de cada par y el service graph sale incompleto. Es la razón por la que el módulo 18 calcula las métricas **antes** del muestreo y el 19 lo mantiene en el mismo Collector. *Fuente: documentación de Grafana «Scale Alloy tail sampling» y OpenTelemetry «Scaling the Collector» (ambas confirmadas por búsqueda; los sitios no abren desde el sandbox).*
10. **El `memory_limiter` rechaza, no pierde: el cliente decide.** Al superar el límite blando (`limit_mib − spike_limit_mib`) el procesador devuelve un error reintentable al receptor; solo se pierden datos si el emisor no reintenta o su cola se llena. Por eso se coloca **el primero** del pipeline, para que la contrapresión llegue al receptor, y su límite se fija por debajo de `GOMEMLIMIT` y del límite del pod. El README recomienda empezar con `spike_limit_mib` ≈ 20 % del límite duro (más si el tráfico es a ráfagas) y fijar `GOMEMLIMIT` coordinado con el limitador; el reparto exacto sale de medir tu tráfico. *Fuente: README oficial del `memorylimiterprocessor` (repositorio `opentelemetry-collector`, leído el 16-oct-2026), que además avisa de que un emisor que no reintente tras el rechazo pierde datos.*

## 15 · ⚠️ Errores comunes

- Confiar en `helm lint` como validación de las configuraciones del Collector/Tempo/Loki.
- Escalar el gateway con tail sampling con un Service normal (sin `load_balancing`).
- Un HPA sin `requests` de CPU (la utilización se calcula sobre requests) o sin metrics-server.
- Olvidar `automountServiceAccountToken: false`, `runAsNonRoot` y `readOnlyRootFilesystem` en cargas propias (el chart `supportops` los fija).
- Usar `kind load docker-image` con la etiqueta `latest` (el kubelet intenta *pull*): usa una etiqueta explícita y `imagePullPolicy: IfNotPresent`.
- Poner la contraseña de Grafana en un values del repositorio.
- Mezclar charts de `grafana/helm-charts` y `grafana-community/helm-charts` sin comprobar cuál mantiene cada componente.

## 16 · Autoevaluación

<details>
<summary>1. (Cálculo) Un HPA con objetivo 70 % de CPU tiene 3 réplicas al 98 % de utilización. ¿Cuántas réplicas pide? ¿Y si estuvieran al 74 %?</summary>

$\lceil 3\cdot 98/70\rceil=\lceil 4,2\rceil=5$. Con 74 %: la razón $74/70=1,057$ está dentro de la tolerancia del 10 % ⇒ **no cambia** (3).

</details>

<details>
<summary>2. (Cálculo) El gateway recibe 150 trazas nuevas/s, `decision_wait` 10 s y cada traza ocupa ≈ 30 KB en memoria. ¿Qué `requests`/`limits` de memoria pones por réplica con 3 réplicas y qué `num_traces`?</summary>

Por réplica: 50 trazas/s × 10 s = 500 trazas × 30 KB ≈ 15 MB de datos; con las estructuras del procesador y GOMEMLIMIT, `requests` 256 Mi y `limits` 512 Mi son holgados. `num_traces` ≥ 500 × 1,5 ≈ 750 por réplica (y mucho más si el tráfico crece: usa 20 000 como en los values y alerta sobre `dropped_too_early`).

</details>

<details>
<summary>3. (Diagnóstico) Tras escalar el gateway de 2 a 4 réplicas aparecen trazas de error «incompletas» en Tempo y el recall de errores baja. El agente usa `load_balancing`. ¿Qué miras?</summary>

(a) La resolución DNS del Service headless (`interval` del resolvedor: hasta que se actualice, parte del tráfico va al anillo viejo); (b) que las réplicas nuevas estén `Ready` antes de entrar en el DNS; (c) que `decision_wait` cubra el retraso; (d) la ventana de estabilización del HPA: cada cambio reasigna trazas en curso. `sampling_trace_dropped_too_early` y `sampling_traces_on_memory` por réplica.

</details>

<details>
<summary>4. (Diagnóstico) `helm install` termina bien pero el pod del Collector está en `CrashLoopBackOff` con `'port' expected type 'string', got unconvertible type 'int'`. ¿Cómo evitas que vuelva a pasar?</summary>

Validar en CI la configuración renderizada con `otelcol-contrib validate` (con las variables de entorno que inyecta el pod) además de `helm lint`/`kubeconform`; y comprobar el tipo en los values (`port: "4317"`).

</details>

<details>
<summary>5. (Transferencia) Tu empresa usa Datadog en producción y quiere un laboratorio OTel/Grafana en Kubernetes para pruebas. ¿Qué parte de este módulo reutilizas y qué cambiarías?</summary>

Reutilizas el patrón agente DaemonSet + gateway Deployment, los values del Collector (cambiando solo el exportador al backend que corresponda) y el CI de validación. Cambiarías los exportadores (`datadog` en contrib) y los datasources; la afinidad por `traceID`, el dimensionado y la validación de configuración se mantienen.

</details>

<details>
<summary>6. (Transferencia) ¿Cuándo elegirías el Operator frente al chart del Collector?</summary>

Cuando necesitas auto-instrumentación por anotación, reconciliación continua o el TargetAllocator para raspar Prometheus a escala con muchos Collectors; si quieres ver y versionar el YAML final y evitar webhooks/cert-manager, el chart con GitOps es más simple de depurar.

</details>

## 17 · Referencias

- Helm: documentación (`helm template`, `helm lint`, esquemas de values, `--wait`): https://helm.sh/docs/ · notas de Helm 4: https://helm.sh/docs/overview/
- OpenTelemetry Helm charts (repositorio verificado en oct-2026: `opentelemetry-collector` 0.176.0 / app 0.162.0; `opentelemetry-operator` 0.126.0 / app 0.161.0; `opentelemetry-kube-stack` 0.24.6): https://github.com/open-telemetry/opentelemetry-helm-charts
- Grafana community charts (Tempo 3.2.0, Loki 18.16.0, Grafana 14.0.0): https://github.com/grafana-community/helm-charts · charts del Pyroscope: https://github.com/grafana/pyroscope/tree/main/operations/pyroscope/helm/pyroscope
- OpenTelemetry Operator: https://github.com/open-telemetry/opentelemetry-operator (CRDs `OpenTelemetryCollector` v1beta1, `Instrumentation` v1alpha1; requisitos de TLS del webhook en el README del chart).
- OpenTelemetry Collector contrib: `loadbalancingexporter` (`load_balancing`) y `tailsamplingprocessor` (README v0.162.0): https://github.com/open-telemetry/opentelemetry-collector-contrib
- Kubernetes: HorizontalPodAutoscaler (algoritmo y `behavior`): https://kubernetes.io/docs/tasks/run-application/horizontal-pod-autoscale/
- kubeconform: https://github.com/yannh/kubeconform · kind: https://kind.sigs.k8s.io · k3s: https://k3s.io